# MD Analysis and MM/PBSA Workflow

## Overview

This notebook presents a reproducible post-processing workflow for Molecular Dynamics (MD) simulations performed using GROMACS. It integrates structural stability analysis with MM/PBSA-based binding free energy estimation for protein–ligand systems.

The workflow is compatible with GPU-accelerated and HPC environments and supports publication-quality figure generation.

## Included Analyses

- RMSD (structural stability)
- RMSF (residue flexibility)
- Radius of Gyration (compactness)
- SASA (solvent exposure)
- Hydrogen Bond Analysis
- DSSP (secondary structure evolution)
- MM/PBSA energy parsing

## Computational Assumptions

- Engine: GROMACS (2021+)
- Trajectory: `.xtc`
- Topology: `.tpr` / `.top`
- Figures exported at 300 dpi

## Workflow

1. Trajectory preprocessing (centering & fitting)
2. Structural stability assessment
3. Flexibility and compactness analysis
4. Intermolecular interaction evaluation
5. Binding free energy estimation

Each module is independent and reproducible.

# Root-Mean-Square Deviation (RMSD) Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import os

plt.rcParams['font.family'] = 'Times New Roman'


# ---------- Helper Function ----------
def read_xvg(filepath):
    """
    Read GROMACS .xvg file.
    Returns time (ns) and RMSD (nm).
    Automatically converts ps to ns if necessary.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"{filepath} not found.")

    time, values = [], []

    with open(filepath, "r") as f:
        for line in f:
            if line.startswith(('@', '#')):
                continue
            parts = line.split()
            if len(parts) >= 2:
                time.append(float(parts[0]))
                values.append(float(parts[1]))

    time = np.array(time)
    values = np.array(values)

    # Convert ps → ns if necessary
    if np.max(time) > 1000:
        time /= 1000.0

    return time, values


# ---------- Optional: Discard Equilibration ----------
def discard_equilibration(time, values, start_ns=0):
    """
    Remove initial equilibration period.
    """
    mask = time >= start_ns
    return time[mask], values[mask]


# ---------- Input Files ----------
rmsd_files = {
    "Complex_1": "rmsd_complex1.xvg",
    "Complex_2": "rmsd_complex2.xvg",
}

equilibration_cutoff = 10  # ns (set 0 to disable)


# ---------- Plot ----------
fig, ax = plt.subplots(figsize=(7, 5), dpi=300)

for label, filepath in rmsd_files.items():
    time, rmsd = read_xvg(filepath)

    if equilibration_cutoff > 0:
        time, rmsd = discard_equilibration(time, rmsd, equilibration_cutoff)

    ax.plot(time, rmsd, linewidth=2.0, label=label)

    # Statistics
    mean_rmsd = np.mean(rmsd)
    std_rmsd = np.std(rmsd)

    print(f"{label}")
    print(f"Mean RMSD: {mean_rmsd:.3f} nm")
    print(f"Std Dev  : {std_rmsd:.3f} nm")
    print("-" * 35)


# ---------- Formatting ----------
ax.set_xlabel("Time (ns)", fontsize=18)
ax.set_ylabel("RMSD (nm)", fontsize=18)

ax.tick_params(axis='both', labelsize=14, direction='in', top=True, right=True)

ax.xaxis.set_minor_locator(MultipleLocator(10))
ax.yaxis.set_minor_locator(MultipleLocator(0.02))
ax.tick_params(axis='both', which='minor', direction='in', length=4)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)

ax.legend(frameon=False, fontsize=12)

fig.tight_layout()


# ---------- Save ----------
output_dir = "../example_outputs"
os.makedirs(output_dir, exist_ok=True)

fig.savefig(os.path.join(output_dir, "rmsd_plot.png"), dpi=300)
fig.savefig(os.path.join(output_dir, "rmsd_plot.pdf"))
fig.savefig(os.path.join(output_dir, "rmsd_plot.svg"))

plt.show()

# Root-Mean-Square Fluctuation (RMSF) Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import os

plt.rcParams['font.family'] = 'Times New Roman'


# ---------- Helper Function ----------
def read_rmsf_xvg(filepath):
    """
    Read GROMACS RMSF .xvg file.
    Returns residue indices and RMSF values (nm).
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"{filepath} not found.")

    residue, rmsf = [], []

    with open(filepath, "r") as f:
        for line in f:
            if line.startswith(('@', '#')):
                continue
            parts = line.split()
            if len(parts) >= 2:
                residue.append(int(float(parts[0])))
                rmsf.append(float(parts[1]))

    return np.array(residue), np.array(rmsf)


# ---------- Optional: Moving Average Smoothing ----------
def smooth_signal(values, window=5):
    """
    Apply moving average smoothing.
    window: number of residues for averaging.
    """
    if window <= 1:
        return values
    return np.convolve(values, np.ones(window)/window, mode='same')


# ---------- Input Files ----------
rmsf_files = {
    "Complex_1": "rmsf_complex1.xvg",
    "Complex_2": "rmsf_complex2.xvg",
}

smoothing_window = 5  # set 1 to disable smoothing


# ---------- Plot ----------
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

for label, filepath in rmsf_files.items():
    residue, rmsf = read_rmsf_xvg(filepath)

    rmsf_processed = smooth_signal(rmsf, smoothing_window)

    ax.plot(residue, rmsf_processed, linewidth=2.0, label=label)

    # Statistics
    mean_rmsf = np.mean(rmsf)
    max_rmsf = np.max(rmsf)

    print(f"{label}")
    print(f"Mean RMSF: {mean_rmsf:.3f} nm")
    print(f"Max  RMSF: {max_rmsf:.3f} nm")
    print("-" * 35)


# ---------- Formatting ----------
ax.set_xlabel("Residue Number", fontsize=18)
ax.set_ylabel("RMSF (nm)", fontsize=18)

ax.tick_params(axis='both', labelsize=14, direction='in', top=True, right=True)

# Automatic limits
ax.set_xlim(min(residue), max(residue))
ax.set_ylim(bottom=0)

# Minor ticks
ax.xaxis.set_minor_locator(MultipleLocator(10))
ax.yaxis.set_minor_locator(MultipleLocator(0.02))
ax.tick_params(axis='both', which='minor', direction='in', length=4)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)

ax.legend(frameon=False, fontsize=12)

fig.tight_layout()


# ---------- Save ----------
output_dir = "../example_outputs"
os.makedirs(output_dir, exist_ok=True)

fig.savefig(os.path.join(output_dir, "rmsf_plot.png"), dpi=300)
fig.savefig(os.path.join(output_dir, "rmsf_plot.pdf"))
fig.savefig(os.path.join(output_dir, "rmsf_plot.svg"))

plt.show()

# Radius of Gyration (Rg) Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import os

plt.rcParams['font.family'] = 'Times New Roman'


# ---------- Helper Function ----------
def read_xvg(filepath):
    """
    Read GROMACS .xvg file.
    Returns time (ns) and values (nm).
    Automatically converts ps to ns if needed.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"{filepath} not found.")

    time, values = [], []

    with open(filepath, "r") as f:
        for line in f:
            if line.startswith(('@', '#')):
                continue
            parts = line.split()
            if len(parts) >= 2:
                time.append(float(parts[0]))
                values.append(float(parts[1]))

    time = np.array(time)
    values = np.array(values)

    # Convert ps → ns if needed
    if np.max(time) > 1000:
        time /= 1000.0

    return time, values


# ---------- Optional: Remove Equilibration ----------
def discard_equilibration(time, values, start_ns=0):
    mask = time >= start_ns
    return time[mask], values[mask]


# ---------- Input Files ----------
Rg_files = {
    "Complex_1": "gyrate_complex1.xvg",
    "Complex_2": "gyrate_complex2.xvg",
}

equilibration_cutoff = 10  # ns (set 0 to disable)


# ---------- Plot ----------
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

for label, filepath in Rg_files.items():
    time, rg = read_xvg(filepath)

    if equilibration_cutoff > 0:
        time, rg = discard_equilibration(time, rg, equilibration_cutoff)

    ax.plot(time, rg, linewidth=2.0, label=label)

    # Statistics
    mean_rg = np.mean(rg)
    std_rg = np.std(rg)

    print(f"{label}")
    print(f"Mean Rg: {mean_rg:.3f} nm")
    print(f"Std Dev: {std_rg:.3f} nm")
    print("-" * 35)


# ---------- Formatting ----------
ax.set_xlabel("Time (ns)", fontsize=18)
ax.set_ylabel("Radius of Gyration (nm)", fontsize=18)

ax.tick_params(axis='both', labelsize=14, direction='in', top=True, right=True)

ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

ax.xaxis.set_minor_locator(MultipleLocator(10))
ax.yaxis.set_minor_locator(MultipleLocator(0.02))
ax.tick_params(axis='both', which='minor', direction='in', length=4)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)

ax.legend(frameon=False, fontsize=12)

fig.tight_layout()


# ---------- Save ----------
output_dir = "../example_outputs"
os.makedirs(output_dir, exist_ok=True)

fig.savefig(os.path.join(output_dir, "rg_plot.png"), dpi=300)
fig.savefig(os.path.join(output_dir, "rg_plot.pdf"))
fig.savefig(os.path.join(output_dir, "rg_plot.svg"))

plt.show()

# Solvent Accessible Surface Area (SASA)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import os

plt.rcParams['font.family'] = 'Times New Roman'


# ---------- Helper Function ----------
def read_xvg(filepath):
    """
    Read GROMACS .xvg file.
    Returns time (ns) and values.
    Automatically converts ps to ns if necessary.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"{filepath} not found.")

    time, values = [], []

    with open(filepath, "r") as f:
        for line in f:
            if line.startswith(('@', '#')):
                continue
            parts = line.split()
            if len(parts) >= 2:
                time.append(float(parts[0]))
                values.append(float(parts[1]))

    time = np.array(time)
    values = np.array(values)

    # Convert ps → ns if needed
    if np.max(time) > 1000:
        time /= 1000.0

    return time, values


# ---------- Optional: Remove Equilibration ----------
def discard_equilibration(time, values, start_ns=0):
    mask = time >= start_ns
    return time[mask], values[mask]


# ---------- Input Files ----------
sasa_files = {
    "Complex_1": "sasa_complex1.xvg",
    "Complex_2": "sasa_complex2.xvg",
}

equilibration_cutoff = 10  # ns (set 0 to disable)


# ---------- Plot ----------
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

for label, filepath in sasa_files.items():
    time, sasa = read_xvg(filepath)

    if equilibration_cutoff > 0:
        time, sasa = discard_equilibration(time, sasa, equilibration_cutoff)

    ax.plot(time, sasa, linewidth=2.0, label=label)

    # Statistics
    mean_sasa = np.mean(sasa)
    std_sasa = np.std(sasa)

    print(f"{label}")
    print(f"Mean SASA: {mean_sasa:.2f} nm²")
    print(f"Std Dev  : {std_sasa:.2f} nm²")
    print("-" * 35)


# ---------- Formatting ----------
ax.set_xlabel("Time (ns)", fontsize=18)
ax.set_ylabel("SASA (nm²)", fontsize=18)

ax.tick_params(axis='both', labelsize=14, direction='in', top=True, right=True)

ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

ax.xaxis.set_minor_locator(MultipleLocator(10))
ax.yaxis.set_minor_locator(MultipleLocator(5))
ax.tick_params(axis='both', which='minor', direction='in', length=4)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)

ax.legend(frameon=False, fontsize=12)

fig.tight_layout()


# ---------- Save ----------
output_dir = "../example_outputs"
os.makedirs(output_dir, exist_ok=True)

fig.savefig(os.path.join(output_dir, "sasa_plot.png"), dpi=300)
fig.savefig(os.path.join(output_dir, "sasa_plot.pdf"))
fig.savefig(os.path.join(output_dir, "sasa_plot.svg"))

plt.show()

# Hydrogen Bond Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams['font.family'] = 'Times New Roman'


# ---------- Helper Function ----------
def read_xvg(filepath):
    """
    Read GROMACS .xvg file.
    Returns time (ns) and hydrogen bond counts.
    Automatically converts ps to ns if necessary.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"{filepath} not found.")

    time, values = [], []

    with open(filepath, "r") as f:
        for line in f:
            if line.startswith(('@', '#')):
                continue
            parts = line.split()
            if len(parts) >= 2:
                time.append(float(parts[0]))
                values.append(float(parts[1]))

    time = np.array(time)
    values = np.array(values)

    if np.max(time) > 1000:
        time /= 1000.0

    return time, values


# ---------- Optional: Remove Equilibration ----------
def discard_equilibration(time, values, start_ns=0):
    mask = time >= start_ns
    return time[mask], values[mask]


# ---------- Input Files ----------
hbond_files = {
    "Complex_1": "hbonds_complex1.xvg",
    "Complex_2": "hbonds_complex2.xvg",
}

equilibration_cutoff = 10  # ns (set 0 to disable)


# ---------- Plot ----------
fig, ax = plt.subplots(figsize=(7, 5), dpi=300)

for label, filepath in hbond_files.items():

    time, hb = read_xvg(filepath)

    if equilibration_cutoff > 0:
        time, hb = discard_equilibration(time, hb, equilibration_cutoff)

    ax.plot(time, hb, linewidth=1.5, label=label)

    # ---- Statistics ----
    mean_hb = np.mean(hb)
    std_hb = np.std(hb)

    # Occupancy: % of frames with ≥1 hydrogen bond
    occupancy = np.sum(hb > 0) / len(hb) * 100

    print(f"{label}")
    print(f"Mean H-bonds : {mean_hb:.2f}")
    print(f"Std Dev      : {std_hb:.2f}")
    print(f"Occupancy    : {occupancy:.1f}%")
    print("-" * 40)


# ---------- Formatting ----------
ax.set_xlabel("Time (ns)", fontsize=18)
ax.set_ylabel("Number of Hydrogen Bonds", fontsize=18)

ax.tick_params(axis='both', labelsize=14, direction='in', top=True, right=True)

ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)

ax.legend(frameon=False, fontsize=12)

fig.tight_layout()


# ---------- Save ----------
output_dir = "../example_outputs"
os.makedirs(output_dir, exist_ok=True)

fig.savefig(os.path.join(output_dir, "hbond_plot.png"), dpi=300)
fig.savefig(os.path.join(output_dir, "hbond_plot.pdf"))
fig.savefig(os.path.join(output_dir, "hbond_plot.svg"))

plt.show()

# Define Secondary Structure of Proteins (DSSP) Analysis

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import dssp
import numpy as np
import matplotlib.pyplot as plt
import os

plt.rcParams['font.family'] = 'Times New Roman'


# ---------- Load Trajectory ----------
topology_file = "md.tpr"
trajectory_file = "md_noPBC.xtc"

u = mda.Universe(topology_file, trajectory_file)

dt_ns = u.trajectory.dt / 1000.0  # convert ps → ns


# ---------- Run DSSP ----------
dssp_analysis = dssp.DSSP(u).run()
ss_array = dssp_analysis.results['dssp']


# ---------- DSSP Groups ----------
helix_codes = {"H", "G", "I"}
sheet_codes = {"E", "B"}
coil_codes  = {"C", "S", "T", "-"}


# ---------- Count Structures Per Frame ----------
helix_counts = []
sheet_counts = []
coil_counts  = []

for frame in ss_array:
    helix_counts.append(sum(ss in helix_codes for ss in frame))
    sheet_counts.append(sum(ss in sheet_codes for ss in frame))
    coil_counts.append(sum(ss in coil_codes for ss in frame))

helix_counts = np.array(helix_counts)
sheet_counts = np.array(sheet_counts)
coil_counts  = np.array(coil_counts)

n_residues = len(ss_array[0])

# Convert counts → percentage
helix_pct = helix_counts / n_residues * 100
sheet_pct = sheet_counts / n_residues * 100
coil_pct  = coil_counts  / n_residues * 100

# Time axis
time = np.arange(len(helix_pct)) * dt_ns


# ---------- Optional: Remove Equilibration ----------
equilibration_cutoff = 10  # ns
mask = time >= equilibration_cutoff

time = time[mask]
helix_pct = helix_pct[mask]
sheet_pct = sheet_pct[mask]
coil_pct  = coil_pct[mask]


# ---------- Statistics ----------
print("Average Secondary Structure Composition:")
print(f"Helix: {np.mean(helix_pct):.2f}%")
print(f"Sheet: {np.mean(sheet_pct):.2f}%")
print(f"Coil : {np.mean(coil_pct):.2f}%")
print("-" * 40)


# ---------- Plot ----------
fig, ax = plt.subplots(figsize=(7, 5), dpi=300)

ax.plot(time, helix_pct, label="Helix (H/G/I)", linewidth=2.0)
ax.plot(time, sheet_pct, label="Sheet (E/B)", linewidth=2.0)
ax.plot(time, coil_pct,  label="Coil/Other", linewidth=2.0)

ax.set_xlabel("Time (ns)", fontsize=18)
ax.set_ylabel("Secondary Structure (%)", fontsize=18)

ax.tick_params(axis='both', labelsize=14, direction='in', top=True, right=True)
ax.set_xlim(left=0)
ax.set_ylim(0, 100)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)

ax.legend(frameon=False, fontsize=12)

fig.tight_layout()


# ---------- Save ----------
output_dir = "../example_outputs"
os.makedirs(output_dir, exist_ok=True)

fig.savefig(os.path.join(output_dir, "dssp_secondary_structure.png"), dpi=300)
fig.savefig(os.path.join(output_dir, "dssp_secondary_structure.pdf"))
fig.savefig(os.path.join(output_dir, "dssp_secondary_structure.svg"))

plt.show()

# MM/PBSA ENERGY PARSING & AGGREGATION

In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np

# ------------------------------------------
# Term Normalisation
# ------------------------------------------

TERM_ALIASES = {
    r'(delta\s*)?g(_|\s|-)*bind(ing)?': 'Delta_G_bind',
    r'(total\s*)?delta\s*g': 'Delta_G_bind',
    r'^total$': 'Delta_G_bind',
    r'^sum$': 'Delta_G_bind',
    r'(vdw|van\s*der\s*waals)': 'vdW',
    r'(ele|coul(omb)?|electrostat(ic)?s?)': 'Ele',
    r'(polar(\s*solv(ation)?)?|gb|pb)': 'Polar_solv',
    r'(np|non[-\s]*polar(\s*solv(ation)?)?|sasa|sa\s*sa)': 'Nonpolar_solv',
    r'(delta\s*e(_|\s|-)*mm)': 'Delta_E_MM',
    r'(delta\s*g(_|\s|-)*solv(ation)?)': 'Delta_G_solv',
}

LINE_RE = re.compile(
    r'^\s*([A-Za-z0-9_/\-\s\(\)\+\.]+?)\s*[:=]\s*'
    r'([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)'
    r'(?:\s*(?:\+/-|\+-|±)\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?))?'
)


def normalise_term(raw_term: str) -> str:
    t = raw_term.strip().lower()
    t = re.sub(r'\s+', ' ', t)
    for pat, canon in TERM_ALIASES.items():
        if re.search(pat, t):
            return canon
    return re.sub(r'[^a-z0-9]+', '_', t).strip('_')


# ------------------------------------------
# Parser
# ------------------------------------------

def parse_dat_file(path: str):
    rows = []

    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = LINE_RE.match(line)
            if not m:
                continue

            raw_term, mean_s, sd_s = m.groups()
            term = normalise_term(raw_term)

            try:
                mean = float(mean_s)
            except Exception:
                continue

            sd = float(sd_s) if sd_s else np.nan

            rows.append({'term': term, 'mean': mean, 'sd': sd})

    return rows


# ------------------------------------------
# Build Wide DataFrame
# ------------------------------------------

def build_energy_dataframe(targets):

    all_rows = []

    for folder, filename in targets:

        path = os.path.join(folder, filename)

        if not os.path.isfile(path):
            fallback = sorted(glob.glob(os.path.join(folder, "*.dat")))
            if fallback:
                path = fallback[0]
                print(f"[NOTE] Using fallback file: {path}")
            else:
                print(f"[WARN] Missing file: {path}")
                continue

        short_id = os.path.basename(folder).split("-")[0]

        parsed = parse_dat_file(path)
        if not parsed:
            print(f"[WARN] No parsable data in: {path}")
            continue

        for r in parsed:
            all_rows.append({'id': short_id, **r})

    if not all_rows:
        return None, None

    df_long = pd.DataFrame(all_rows)

    # Wide format
    mean_w = df_long.pivot_table(index='id', columns='term', values='mean')
    sd_w   = df_long.pivot_table(index='id', columns='term', values='sd')

    ordered_terms = [
        'vdW',
        'Ele',
        'Polar_solv',
        'Nonpolar_solv',
        'Delta_E_MM',
        'Delta_G_solv',
        'Delta_G_bind'
    ]

    combined = pd.DataFrame(index=mean_w.index)

    for term in ordered_terms:
        if term in mean_w.columns:
            combined[f"{term}_mean"] = mean_w[term]
            combined[f"{term}_sd"]   = sd_w[term]

    combined = combined.reset_index()

    return df_long, combined


# ------------------------------------------
# Targets
# ------------------------------------------

TARGETS = [
    ("Complex_1", "summary_energy.dat"),
    ("Complex_2", "summary_energy.dat"),
]


df_long, df_wide = build_energy_dataframe(TARGETS)

if df_wide is None:
    print("No valid MM/PBSA data found.")
else:

    # Save outputs
    df_long.to_csv("energies_long.csv", index=False)
    df_wide.to_csv("energies_wide.csv", index=False)

    print("Saved:")
    print(" - energies_long.csv")
    print(" - energies_wide.csv")
    print("Shape:", df_wide.shape)

    display(df_wide)

# PER-RESIDUE MM/PBSA (LINE "PEAK" PLOT — MULTI-COMPLEX)

In [ ]:
# ==========================================================
# • Assumes g_mmpbsa output (kJ/mol, no conversion)
# • Window, thresholds, and styling fully controlled
# ==========================================================

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Times New Roman'


# ------------------- Global Settings -------------------
UNITS = "kJ/mol"
XMIN, XMAX = 324, 610
YMIN, YMAX = -6.0, 6.0
YTICK = 2.0

TOP_K_EACH = 12
MIN_ABS = 0.15
MIN_SEP = 3
MAX_LABELS = 8

LABEL_FONTSIZE = 13
LABEL_OFFSET_POS = 0.25
LABEL_OFFSET_NEG = -0.30
DRAW_ZEROLINE = False


COMPLEX_CONFIG = {
    "Complex_1": {
        "folder": "DB16447-ligand_ctrl",
        "color": "red",
        "panel_label": "(a)"
    },
    "Complex_2": {
        "folder": "CHEMBL221598-ligand1",
        "color": "green",
        "panel_label": "(b)"
    }
}


# ------------------- Utilities -------------------

def apply_pub_style(ax, tick_labelsize=12):
    ax.tick_params(axis='both', which='major',
                   labelsize=tick_labelsize, width=2, length=7,
                   direction='in', top=True, right=True)
    ax.minorticks_off()
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_linewidth(2.0)
        s.set_edgecolor('black')


def even_ticks_with_zero(ymin, ymax, step):
    start = np.floor(ymin/step)*step
    end   = np.ceil(ymax/step)*step
    ticks = np.arange(start, end + 0.1*step, step)
    if ymin <= 0 <= ymax and 0 not in ticks:
        ticks = np.sort(np.append(ticks, 0.0))
    return ticks


def read_final_contrib_energy(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.strip() or line.lstrip().startswith("#"):
                continue

            toks = line.split()
            res_token = toks[0]
            nums = [float(t) for t in toks[1:] if re.match(r'^-?\d+\.?\d*$', t)]

            if len(nums) < 7:
                continue

            m = re.search(r"([A-Za-z]{1,3})[-_\s]*?(\d+)", res_token)
            resname = m.group(1).title()
            resid = int(m.group(2))

            total = nums[6]
            rows.append((resid, resname, total))

    df = pd.DataFrame(rows, columns=["resid","resname","Total"])
    return df.sort_values("resid").reset_index(drop=True)


def select_residues(perres):
    cand = perres.loc[perres["Total"].abs() >= MIN_ABS].copy()
    stab = cand.nsmallest(TOP_K_EACH, "Total")
    dest = cand.nlargest(TOP_K_EACH, "Total")
    merged = pd.concat([stab, dest])

    selected = []
    for _, row in merged.sort_values("Total", key=lambda s: s.abs(), ascending=False).iterrows():
        if all(abs(row["resid"] - r["resid"]) >= MIN_SEP for r in selected):
            selected.append(row)
        if len(selected) >= MAX_LABELS:
            break

    return pd.DataFrame(selected)


# ------------------- Main Loop -------------------

for name, cfg in COMPLEX_CONFIG.items():

    folder = cfg["folder"]
    color = cfg["color"]
    panel_label = cfg["panel_label"]

    path = os.path.join(folder, "final_contrib_energy.dat")
    perres = read_final_contrib_energy(path)

    # Remove ligand entries
    perres = perres[~perres["resname"].isin({"Unl","Lig","Ligand"})]
    perres = perres[(perres["resid"] >= XMIN) & (perres["resid"] <= XMAX)]

    perres["Label"] = perres["resname"] + "-" + perres["resid"].astype(str)

    sel = select_residues(perres)
    sel_res = sel["resid"].to_numpy()

    # Plot
    fig, ax = plt.subplots(figsize=(9.5, 6.0), dpi=300)

    if DRAW_ZEROLINE:
        ax.axhline(0, linewidth=1.2)

    ax.plot(perres["resid"], perres["Total"],
            color=color, linewidth=1.8)

    for _, row in sel.iterrows():
        yi = row["Total"]
        dy = LABEL_OFFSET_POS if yi >= 0 else LABEL_OFFSET_NEG
        va = "bottom" if yi >= 0 else "top"
        ax.text(row["resid"], yi + dy,
                row["Label"],
                fontsize=LABEL_FONTSIZE,
                ha='center', va=va,
                rotation=90)

    ax.set_xlabel("Residue Number", fontsize=20)
    ax.set_ylabel(f"Contribution Energy ({UNITS})", fontsize=20)
    ax.set_xlim(XMIN, XMAX)
    ax.set_ylim(YMIN, YMAX)
    ax.set_yticks(even_ticks_with_zero(YMIN, YMAX, YTICK))
    ax.text(0.98, 0.95, f"{panel_label} {name}",
            transform=ax.transAxes,
            fontsize=15, ha="right", va="top")

    apply_pub_style(ax)
    fig.tight_layout()

    basename = f"per_residue_line_{name}"
    for ext in ("png", "svg", "pdf"):
        fig.savefig(f"{basename}.{ext}", dpi=300, bbox_inches="tight")

    print(f"\n{name} — Informative Residues:")
    summary = sel[["resid","resname","Total"]].copy()
    summary.columns = ["Resid","ResName",f"ΔG_res ({UNITS})"]
    print(summary.sort_values(f"ΔG_res ({UNITS})").to_string(index=False))

# THE END